# Week 7 — Advanced Models

* Try Gradient Boosting (e.g., XGBoost or LightGBM).
* Perform light hyperparameter tuning (depth, learning rate, n_estimators).
* Deliverable: 05_advanced_models.ipynb with test metrics.

In [98]:
from pathlib import Path
import geopandas as gpd
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV, ParameterGrid, cross_val_score
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
from xgboost import XGBRegressor

In [100]:
data_dir = Path("/Users/jingying/Documents/IDX Exchange/py/cleaned_data/run_20261014_180636_999980")
district_file = Path("/Users/jingying/Documents/IDX Exchange/py/DistrictAreas2425/DistrictAreas2425.shp")

y_train = pd.read_csv(data_dir / "y_train.csv", index_col="row_id")["ClosePrice"]
y_test = pd.read_csv(data_dir / "y_test.csv", index_col="row_id")["ClosePrice"]
settings = joblib.load(data_dir / "preprocessing_and_baseline.joblib")
#print(settings)

raw_num = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude"]
raw_cat = [col for col in settings["schema"]["categorical"] if col != "GeoCluster"]
history = pd.read_csv(data_dir / "quality_cleaned_history.csv", index_col="row_id", 
                      usecols=["row_id", "CloseMonth", "ClosePrice"] + raw_num + raw_cat, 
                      dtype=dict.fromkeys(raw_cat, str),)
history["CloseMonth"] = pd.PeriodIndex(history["CloseMonth"], freq="M")
train_months = history.loc[y_train.index, "CloseMonth"]
train_start = train_months.min()
validation_month = train_months.max()
cv_months = pd.period_range(end=validation_month, periods=3, freq="M")
test_month = history.loc[y_test.index, "CloseMonth"].iloc[0]
print("Cross-Validation months:", ", ".join(map(str, cv_months)))
print("Test month:", test_month)

Cross-Validation months: 2026-01, 2026-02, 2026-03
Test month: 2026-04


In [114]:
districts = gpd.read_file(district_file)[["CDCode", "geometry"]]
locations = history[["Longitude", "Latitude"]].dropna()
points = gpd.GeoDataFrame(index=locations.index,
                          geometry=gpd.points_from_xy(locations["Longitude"], locations["Latitude"]),
                          crs="EPSG:4326").to_crs(districts.crs)
matches = points.sjoin(districts, how="inner", predicate="within")
districts_by_sale = matches.groupby(level="row_id")["CDCode"].agg(set).to_dict()

def add_districts(X_fit, X_eval, fit_ids, eval_ids):
    encoder = MultiLabelBinarizer(sparse_output=True)
    fit_codes = encoder.fit_transform([districts_by_sale.get(i, ()) for i in fit_ids])
    #eval_codes = encoder.transform([districts_by_sale.get(i, ()) for i in eval_ids])
    # Encode only districts observed in the training split.
    known_districts = set(encoder.classes_)
    eval_codes = encoder.transform([
        known_districts.intersection(districts_by_sale.get(i, ())) for i in eval_ids
    ])
    return (sparse.hstack([X_fit, fit_codes], format="csr"),
            sparse.hstack([X_eval, eval_codes], format="csr"))

In [115]:
earlier = history.loc[history["CloseMonth"].between(train_months.min(), validation_month - 1)]
validation = history.loc[history["CloseMonth"].eq(validation_month)]
low, high = earlier["ClosePrice"].quantile([0.005, 0.995])
fit_sales = earlier.loc[earlier["ClosePrice"].between(low, high)]
rows = pd.concat([fit_sales, validation]).copy()

# Recreate the Week 3 numeric features at the month of valuation.
month = rows["CloseMonth"].dt.month
year = rows["CloseMonth"].dt.year
rows["PropertyAge"] = year - rows["YearBuilt"]
rows["BedBathRatio"] = rows["BedroomsTotal"] / rows["BathroomsTotalInteger"].replace(0, np.nan)
rows["MonthSin"] = np.sin(2 * np.pi * month / 12)
rows["MonthCos"] = np.cos(2 * np.pi * month / 12)
rows["TimeIndex"] = (year - 2020) * 12 + month - 1

# Use the previous 12 months. Prefer ZIP, then county, then the overall median.
ppsf = earlier["ClosePrice"] / earlier["LivingArea"]
reference = earlier.assign(PPSF=ppsf).loc[ppsf.gt(0) & np.isfinite(ppsf)]
rows["HistoricPPSF"] = np.nan
for target_month, sales in rows.groupby("CloseMonth"):
    prior = reference.loc[reference["CloseMonth"].between(target_month - 12, target_month - 1)]
    values = pd.Series(np.nan, index=sales.index)
    for area in ["PostalCode", "CountyOrParish"]:
        stats = prior.groupby(area)["PPSF"].agg(["count", "median"])
        local_median = stats.loc[stats["count"] >= 5, "median"]
        values = values.fillna(sales[area].map(local_median))
    rows.loc[sales.index, "HistoricPPSF"] = values.fillna(prior["PPSF"].median())

In [116]:
# Clone keeps the Week 3 settings and lets this split learn its own parameters.
coord_cols = ["Latitude", "Longitude"]
geo = clone(settings["geo"])
geo.fit(fit_sales[coord_cols].dropna())
observed = rows[coord_cols].notna().all(axis=1)
rows["GeoCluster"] = "unknown"
rows.loc[observed, "GeoCluster"] = geo.predict(rows.loc[observed, coord_cols]).astype(str)

for col in settings["feature_cols"]:
    if col.endswith("_missing"):
        rows[col] = rows[col.removesuffix("_missing")].isna().astype(int)

features = rows[settings["feature_cols"]]
preprocessor = clone(settings["pipeline"].named_steps["preprocess"])
X_fit = preprocessor.fit_transform(features.loc[fit_sales.index])
X_valid = preprocessor.transform(features.loc[validation.index])
X_fit, X_valid = add_districts(X_fit, X_valid, fit_sales.index, validation.index)
y_fit = fit_sales["ClosePrice"].to_numpy()
y_valid = validation["ClosePrice"].to_numpy()

In [121]:
grid = {
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.10],
    "n_estimators": [150, 300, 500, 1000],
}
model_settings = dict(objective="reg:squarederror", tree_method="hist", random_state=42, n_jobs=2)

trials = []
for params in ParameterGrid(grid):
    model = XGBRegressor(**model_settings, **params)
    model.fit(X_fit, y_fit)
    predicted = model.predict(X_valid)
    mdape = 100 * np.median(np.abs(y_valid - predicted) / y_valid)
    trials.append({**params, "Validation MdAPE (%)": mdape})

# In a tie, prefer the shallower model, then the one with fewer trees.
ranked = pd.DataFrame(trials).sort_values(
    ["Validation MdAPE (%)", "max_depth", "n_estimators", "learning_rate"]
)
best_params = {key: trials[ranked.index[0]][key] for key in grid}
display(ranked.reset_index(drop=True).round(3))
print("Selected parameters:", best_params)

,learning_rate,max_depth,n_estimators,Validation MdAPE (%)
0,0.10,8,1000,8.433
1,0.05,8,1000,8.660
2,0.10,8,500,8.724
3,0.10,6,1000,8.779
4,0.10,8,300,9.041
5,0.05,8,500,9.131
6,0.05,6,1000,9.187
7,0.10,6,500,9.280
8,0.05,8,300,9.358
9,0.10,8,150,9.381


Selected parameters: {'max_depth': 8, 'learning_rate': 0.1, 'n_estimators': 1000}


In [122]:
X_train, X_test = add_districts(
    sparse.load_npz(run_dir / "X_train.npz"), sparse.load_npz(run_dir / "X_test.npz"),
    y_train.index, y_test.index,
)
final_model = XGBRegressor(**model_settings, **best_params)
final_model.fit(X_train, y_train)
test_predictions = final_model.predict(X_test)

In [127]:
range_ids = pd.read_csv(run_dir / "y_test_avm_range.csv", usecols=["row_id"])["row_id"]
test_groups = {
    "Full test rows": np.ones(len(y_test), dtype=bool),
    "Test rows within training price bounds": y_test.index.isin(range_ids),
}

results = []
for group, mask in test_groups.items():
    actual = y_test.to_numpy()[mask]
    predicted = test_predictions[mask]
    ape = 100 * np.abs(actual - predicted) / actual
    results.append({
        "Model": "XGBoost", "Test month": str(test_month), "Test sample": group,
        "Sales": len(actual), "R2": r2_score(actual, predicted),
        "MAE ($)": mean_absolute_error(actual, predicted),
        "MAPE (%)": ape.mean(), "MdAPE (%)": np.median(ape),
    })

test_metrics = pd.DataFrame(results)
test_metrics.to_csv(run_dir / "week7_test_metrics.csv", index=False)
display(test_metrics.round(4))

,Model,Test month,Test sample,Sales,R2,MAE ($),MAPE (%),MdAPE (%)
0,XGBoost,2026-04,Full test rows,12458,0.0094,387331.9954,13.1265,8.4608
1,XGBoost,2026-04,Test rows within training price bounds,12321,0.8970,160774.5648,12.2989,8.3639
